In [1]:
import pandas as pd
import altair as alt

euro_df = pd.read_csv('../data/fully_cleaned_top_spotify_songs_EU.csv')

In [2]:
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'popularity']

country_names = {
    'BE': 'Belgium', 'DE': 'Germany', 'ES': 'Spain', 'FR': 'France', 'GB': 'United Kingdom',
    'IT': 'Italy', 'NL': 'Netherlands', 'PL': 'Poland', 'PT': 'Portugal', 'SE': 'Sweden'
}

country_avg = euro_df.groupby('country')[features].mean()

# normalize each feature 0-1 so colors are comparable across columns
country_norm = (country_avg - country_avg.min()) / (country_avg.max() - country_avg.min())

country_norm['country_name'] = country_norm.index.map(country_names)

# also keep raw values for tooltips
country_avg['country_name'] = country_avg.index.map(country_names)

# melt both into long format
norm_long = country_norm.melt(id_vars='country_name', var_name='feature', value_name='normalized')
raw_long = country_avg.melt(id_vars='country_name', var_name='feature', value_name='raw_value')

heatmap_df = norm_long.merge(raw_long, on=['country_name', 'feature'])
heatmap_df

,country_name,feature,normalized,raw_value
0,Belgium,danceability,0.630655,0.702728
1,Germany,danceability,0.536410,0.689004
2,Spain,danceability,0.427661,0.673168
3,France,danceability,1.000000,0.756512
4,United Kingdom,danceability,0.310218,0.656066
...,...,...,...,...
95,Italy,popularity,0.083866,73.084000
96,Netherlands,popularity,0.445468,76.930000
97,Poland,popularity,0.001316,72.206000
98,Portugal,popularity,0.618842,78.774000


In [ ]:
chart = alt.Chart(heatmap_df).mark_rect().encode(
    x=alt.X('feature:N', title='Audio Feature'),
    y=alt.Y('country_name:N', title='Country'),
    color=alt.Color('normalized:Q', scale=alt.Scale(scheme='yellowgreenblue'), title='Normalized Value'),
    tooltip=[
        alt.Tooltip('country_name:N', title='Country'),
        alt.Tooltip('feature:N', title='Feature'),
        alt.Tooltip('raw_value:Q', title='Actual Value', format='.3f'),
        alt.Tooltip('normalized:Q', title='Normalized', format='.3f')
    ]
).properties(
    title='Average Audio Features by Country (Normalized)',
    width=450,
    height=350
)

chart.show()

In [4]:
chart.save('heatmap.html')